# 03 Project: Trading Practice
Interview-style SQL practice on a simulated trading app: about 3,000 users, 16,500 trades, 20 assets
(crypto, stocks, ETFs, precious metals). Fixed random data, so every run gives the same results. No real customer data.

Run the **two setup cells** once, then answer each task in its `%%sql` cell.

**Tables**
- `users(user_id, country, signup_date)`
- `trades(trade_id, user_id, asset_id, traded_at, amount_eur)`
- `assets(asset_id, symbol, asset_class)`, asset_class is 'crypto', 'stock', 'etf' or 'metal'

**The 7 questions before every query**
1. What does one row of the result look like?
2. Which tables hold the columns I need?
3. Must one table stay complete? (LEFT) or only matches? (INNER)
4. Which rows do I throw away first? (WHERE)
5. Do I collapse rows into totals? (GROUP BY)
6. Do I filter on those totals? (HAVING)
7. Order? (ORDER BY)

Fewer rows than the source → GROUP BY. Same rows plus a new column → OVER.

Engine: **SQLite**. Dates are text like `'2026-10-15 13:45:00'`, so `traded_at >= '2026-10-01'` works.

In [1]:
# SETUP: run once. Builds data\trading_practice.db (re-running rebuilds the same data)
import sqlite3, random
from datetime import datetime, timedelta

random.seed(42)
con = sqlite3.connect("../data/trading_practice.db")
cur = con.cursor()
cur.executescript("""
DROP TABLE IF EXISTS trades; DROP TABLE IF EXISTS users; DROP TABLE IF EXISTS assets;
CREATE TABLE users  (user_id INTEGER PRIMARY KEY, country TEXT, signup_date TEXT);
CREATE TABLE assets (asset_id INTEGER PRIMARY KEY, symbol TEXT, asset_class TEXT);
CREATE TABLE trades (trade_id INTEGER PRIMARY KEY, user_id INTEGER, asset_id INTEGER,
                     traded_at TEXT, amount_eur REAL);
""")

assets = [("BTC","crypto"),("ETH","crypto"),("SOL","crypto"),("XRP","crypto"),("ADA","crypto"),
          ("DOGE","crypto"),("AAPL","stock"),("MSFT","stock"),("TSLA","stock"),("NVDA","stock"),
          ("SAP","stock"),("OMV","stock"),("VWCE","etf"),("IWDA","etf"),("EUNL","etf"),
          ("XAU","metal"),("XAG","metal"),("XPT","metal"),("XPD","metal"),("SPYI","etf")]
cur.executemany("INSERT INTO assets VALUES (?,?,?)",
                [(i+1, s, c) for i, (s, c) in enumerate(assets)])
by_class = {}
for i, (s, c) in enumerate(assets):
    by_class.setdefault(c, []).append(i+1)

countries = ["AT"]*35 + ["DE"]*30 + ["FR"]*10 + ["IT"]*10 + ["ES"]*8 + ["SK"]*7
start = datetime(2024, 1, 1)
users = []
for uid in range(1, 3001):
    signup = start + timedelta(days=random.randint(0, 1040))
    users.append((uid, random.choice(countries), signup.strftime("%Y-%m-%d")))
cur.executemany("INSERT INTO users VALUES (?,?,?)", users)

end = datetime(2026, 11, 10)
trades, tid = [], 1
for uid, country, signup in users:
    if random.random() < 0.18:        # some users never trade
        continue
    s = datetime.strptime(signup, "%Y-%m-%d")
    n = random.choice([1, 1, 2, 3, 5, 8, 13, 20])
    crypto_first = random.random() < 0.65
    for k in range(n):
        if k == 0:
            cls = "crypto" if crypto_first else random.choice(["stock", "etf", "metal"])
        else:
            cls = random.choices(["crypto", "stock", "etf", "metal"], [55, 25, 15, 5])[0]
        ts = s + timedelta(days=random.randint(0, max(1, (end - s).days)),
                           seconds=random.randint(0, 86399))
        if k == 0:
            ts = s + timedelta(days=random.randint(0, 20), seconds=random.randint(0, 86399))
        if ts > end:
            ts = end - timedelta(seconds=random.randint(0, 86399))
        amt = round(random.lognormvariate(5, 1.1), 2)
        trades.append((tid, uid, random.choice(by_class[cls]), ts.strftime("%Y-%m-%d %H:%M:%S"), amt))
        tid += 1
cur.executemany("INSERT INTO trades VALUES (?,?,?,?,?)", trades)
con.commit()

con.close()
print(f"Built: {len(users)} users, {len(trades)} trades, {len(assets)} assets")


Built: 3000 users, 16512 trades, 20 assets


In [2]:
# turns on the %%sql cell magic
%load_ext sql
# opens data\trading_practice.db (built by the cell above)
%sql sqlite:///../data/trading_practice.db
# show up to 20 rows per result (default is 10)
%config SqlMagic.displaylimit = 20

Connecting to 'sqlite:///../data/trading_practice.db'

## Warm-up
How many rows does each table have?

*Answer the 7 questions yourself first, then open the hint.*

<details><summary>Hint: the 7 answers</summary>

1. **One row:** a single row with three counts
2. **Tables:** trades, users, assets, each counted on its own
3. **Join:** none. Three scalar subqueries `(SELECT COUNT(*) FROM …)` side by side
4. **WHERE:** none
5. **GROUP BY:** no
6. **HAVING:** no
7. **ORDER BY:** no, only one row

</details>

In [3]:
%%sql
SELECT
  (SELECT COUNT(*) FROM trades) AS num_trades,
  (SELECT COUNT(*) FROM users)  AS num_users,
  (SELECT COUNT(*) FROM assets) AS num_assets

Running query in 'sqlite:///../data/trading_practice.db'

num_trades,num_users,num_assets
16512,3000,20


## Task 1
Total trading volume (`SUM(amount_eur)`) and number of trades **per country** in **October 2026**, highest volume first.

*Answer the 7 questions yourself first, then open the hint.*

<details><summary>Hint: the 7 answers</summary>

1. **One row:** one country, with its trade count and volume
2. **Tables:** trades (amount, date) + users (country)
3. **Join:** INNER. Only users who have trades matter
4. **WHERE:** `traded_at >= '2026-10-01' AND traded_at < '2026-11-01'` (start included, end excluded)
5. **GROUP BY:** country
6. **HAVING:** no
7. **ORDER BY:** volume DESC

</details>

In [4]:
%%sql
SELECT u.country, COUNT(*) AS num_trades, SUM(t.amount_eur) AS total_volume
FROM trades t
JOIN users u ON u.user_id = t.user_id
WHERE t.traded_at >= '2026-10-01' AND t.traded_at < '2026-11-01'
GROUP BY u.country
ORDER BY total_volume DESC

Running query in 'sqlite:///../data/trading_practice.db'

country,num_trades,total_volume
AT,562,154654.6
DE,516,128205.03
SK,185,51294.0
IT,169,42436.54
FR,153,39381.97
ES,131,35800.17


## Task 2
Users who **signed up in 2026** but have **never traded**.

*Answer the 7 questions yourself first, then open the hint.*

<details><summary>Hint: the 7 answers</summary>

1. **One row:** one user (id, signup date)
2. **Tables:** users + trades
3. **Join:** LEFT. users must stay complete, non-traders have no match in trades
4. **WHERE:** signup in 2026 **and** `t.trade_id IS NULL` (no match = never traded)
5. **GROUP BY:** no
6. **HAVING:** no
7. **ORDER BY:** optional

</details>

In [5]:
%%sql
SELECT u.user_id, u.signup_date
FROM users u
LEFT JOIN trades t ON u.user_id = t.user_id
WHERE u.signup_date >= '2026-01-01' AND u.signup_date < '2027-01-01'
  AND t.trade_id IS NULL

Running query in 'sqlite:///../data/trading_practice.db'

user_id,signup_date
36,2026-01-17
58,2026-03-28
80,2026-11-05
93,2026-08-31
95,2026-05-16
110,2026-09-21
129,2026-06-26
134,2026-09-12
140,2026-09-23
183,2026-09-27


## Task 3
Each user's **first trade**: user_id, the date, and the **asset class** of that trade.

*Answer the 7 questions yourself first, then open the hint.*

<details><summary>Hint: the 7 answers</summary>

1. **One row:** one user, with first trade date and asset class
2. **Tables:** trades + assets (asset_class)
3. **Join:** INNER. Only users who traded
4. **WHERE:** `rn = 1`, after numbering each user's trades by date
5. **GROUP BY:** no. Same rows plus a new column → `ROW_NUMBER() OVER (PARTITION BY user_id ORDER BY traded_at)`
6. **HAVING:** no
7. **ORDER BY:** optional

</details>

In [6]:
%%sql
WITH ranked AS (
  SELECT t.user_id,
         t.traded_at,
         t.asset_id,
         ROW_NUMBER() OVER (PARTITION BY t.user_id ORDER BY t.traded_at) AS rn
  FROM trades t
)
SELECT r.user_id, r.traded_at AS first_trade_at, a.asset_class
FROM ranked r
JOIN assets a ON a.asset_id = r.asset_id
WHERE r.rn = 1

Running query in 'sqlite:///../data/trading_practice.db'

user_id,first_trade_at,asset_class
1,2024-08-17 06:54:44,crypto
2,2025-07-29 06:30:45,crypto
4,2024-08-02 12:12:14,crypto
5,2024-06-27 22:40:00,crypto
6,2026-05-31 10:07:32,crypto
9,2026-11-09 11:56:41,crypto
10,2024-02-25 03:23:26,metal
11,2025-02-12 03:58:46,crypto
12,2026-05-28 04:15:15,crypto
13,2026-07-14 03:22:17,crypto


## Task 4
Assets traded by **more than 500 different users**, with the number of users.

*Answer the 7 questions yourself first, then open the hint.*

<details><summary>Hint: the 7 answers</summary>

1. **One row:** one asset, with its number of distinct users
2. **Tables:** assets (symbol) + trades (user_id)
3. **Join:** INNER
4. **WHERE:** none
5. **GROUP BY:** asset (symbol)
6. **HAVING:** `COUNT(DISTINCT t.user_id) > 500`
7. **ORDER BY:** user count DESC

</details>

In [7]:
%%sql
SELECT a.symbol, COUNT(DISTINCT t.user_id) AS n_users
FROM assets a
JOIN trades t ON a.asset_id = t.asset_id
GROUP BY a.symbol
HAVING COUNT(DISTINCT t.user_id) > 500   -- repeat the aggregate: aliases in HAVING only work in SQLite/MySQL
ORDER BY n_users DESC

Running query in 'sqlite:///../data/trading_practice.db'

symbol,n_users
BTC,1017
ETH,1009
XRP,987
SOL,985
DOGE,982
ADA,940
OMV,542
SAP,520
TSLA,518
AAPL,512


## Task 5
For each **asset class**, the number of **different users** who traded it in **2026**, but only classes with **more than 1,000** users.

*Answer the 7 questions yourself first, then open the hint.*

<details><summary>Hint: the 7 answers</summary>

1. **One row:** one asset class, with its number of distinct users
2. **Tables:** assets (asset_class) + trades
3. **Join:** INNER
4. **WHERE:** traded in 2026 (`>= '2026-01-01' AND < '2027-01-01'`)
5. **GROUP BY:** asset_class
6. **HAVING:** `COUNT(DISTINCT t.user_id) > 1000`
7. **ORDER BY:** user count DESC

</details>

In [8]:
%%sql
SELECT a.asset_class, COUNT(DISTINCT t.user_id) AS n_users
FROM assets a
JOIN trades t ON a.asset_id = t.asset_id
WHERE t.traded_at >= '2026-01-01' AND t.traded_at < '2027-01-01'
GROUP BY a.asset_class
HAVING COUNT(DISTINCT t.user_id) > 1000
ORDER BY n_users DESC

Running query in 'sqlite:///../data/trading_practice.db'

asset_class,n_users
crypto,1583
stock,1106


## Task 6
**Monthly** trading volume in 2026, one row per month.

*Answer the 7 questions yourself first, then open the hint.*

<details><summary>Hint: the 7 answers</summary>

1. **One row:** one month, with volume and trade count
2. **Tables:** trades only
3. **Join:** none
4. **WHERE:** traded in 2026
5. **GROUP BY:** month. `substr(traded_at, 1, 7)` turns '2026-10-15 13:45:00' into '2026-10'
6. **HAVING:** no
7. **ORDER BY:** month

</details>

In [9]:
%%sql
SELECT substr(t.traded_at, 1, 7) AS month,
       SUM(t.amount_eur)         AS volume_eur,
       COUNT(*)                  AS n_trades
FROM trades t
WHERE t.traded_at >= '2026-01-01' AND t.traded_at < '2027-01-01'
GROUP BY month
ORDER BY month

Running query in 'sqlite:///../data/trading_practice.db'

month,volume_eur,n_trades
2026-01,159501.67,586
2026-02,158186.58,563
2026-03,191607.12,718
2026-04,191571.29,715
2026-05,223373.8,819
2026-06,231374.3,859
2026-07,267131.82,1030
2026-08,318944.96,1162
2026-09,372454.49,1318
2026-10,451772.31,1716


## Task 7
Of the users whose **first** trade was **crypto**, how many later bought a **stock**? Give both numbers and the conversion rate. A typical fintech interview question.

*Answer the 7 questions yourself first, then open the hint.*

<details><summary>Hint: the 7 answers</summary>

1. **One row:** one summary row: crypto-first users, how many of them bought a stock, and the rate in %
2. **Tables:** trades + assets
3. **Join:** first INNER trades → assets to rank trades, then LEFT crypto-first users → stock buyers (keep all crypto-first users, count the matches)
4. **WHERE:** `rn = 1 AND asset_class = 'crypto'` for the first group, `asset_class = 'stock'` for the second
5. **GROUP BY:** no. Aggregates over the whole result give one row
6. **HAVING:** no
7. **ORDER BY:** no. *Follow-up: why no date check for “later”? Their first trade was crypto, so any stock trade comes after it.*

</details>

In [10]:
%%sql
WITH ranked AS (
  SELECT t.user_id, a.asset_class, t.traded_at,
         ROW_NUMBER() OVER (PARTITION BY t.user_id ORDER BY t.traded_at) AS rn
  FROM trades t
  JOIN assets a ON a.asset_id = t.asset_id
),
crypto_first AS (
  SELECT user_id FROM ranked
  WHERE rn = 1 AND asset_class = 'crypto'
),
stock_buyers AS (
  SELECT DISTINCT t.user_id
  FROM trades t
  JOIN assets a ON a.asset_id = t.asset_id
  WHERE a.asset_class = 'stock'
)
SELECT COUNT(*)         AS crypto_first_users,
       COUNT(s.user_id) AS later_bought_stock,
       ROUND(100.0 * COUNT(s.user_id) / COUNT(*), 1) AS conversion_pct   -- 100.0 forces decimal division
FROM crypto_first c
LEFT JOIN stock_buyers s ON s.user_id = c.user_id

Running query in 'sqlite:///../data/trading_practice.db'

crypto_first_users,later_bought_stock,conversion_pct
1554,798,51.4


## Task 8
Count trades by size: **small** (under 100 €), **medium** (100 to 999 €), **large** (1,000 € and more), most common first.

*Answer the 7 questions yourself first, then open the hint.*

<details><summary>Hint: the 7 answers</summary>

1. **One row:** one size bucket, with its trade count
2. **Tables:** trades only
3. **Join:** none
4. **WHERE:** none
5. **GROUP BY:** the bucket, built with `CASE WHEN … THEN … ELSE … END`. First true `WHEN` wins
6. **HAVING:** no
7. **ORDER BY:** trade count DESC

</details>

In [11]:
%%sql
SELECT CASE WHEN t.amount_eur < 100  THEN 'small'
            WHEN t.amount_eur < 1000 THEN 'medium'
            ELSE 'large'
       END AS bucket,
       COUNT(*) AS n_trades
FROM trades t
GROUP BY bucket
ORDER BY n_trades DESC

Running query in 'sqlite:///../data/trading_practice.db'

bucket,n_trades
medium,9886
small,5976
large,650


## My mistakes

**1. Date range with `<=` on text dates.** I wrote `traded_at <= '2026-11-01'` for October.
Dates are text, and `'2026-11-01 00:00:00'` is *longer* than `'2026-11-01'`, so it counts as bigger.
The same trap with `<= '2026-10-31'` would silently drop every trade on 31 October.
Rule: `>= first day AND < first day of the next period`.

**2. Alias in `HAVING`.** `HAVING n_users > 500` works in SQLite and MySQL, but is an error in PostgreSQL and SQL Server.
Rule: repeat the aggregate, `HAVING COUNT(DISTINCT t.user_id) > 500`.